# Module 1: Bronze layer, ingesting the raw transactions

**Project:** Dubai Real Estate Analytics · **Layer:** Bronze · **Time:** 60–90 minutes

By the end of this notebook you will have:
1. Read 1.16 GB of CSV (1,814,844 rows) with PySpark
2. Added two audit columns that record where each row came from and when it was loaded
3. Saved everything as a **Delta table** called `workspace.dubai_re.bronze_transactions`
4. Queried it with SQL in the same notebook

**How this notebook works**
- 🧠 **Why we're doing this** boxes explain the concept, link it to SQL or Excel you already know, and give you the interview angle.
- ✏️ **TODO** cells are yours. Replace every `___` with code. If you see `NameError: name '___' is not defined`, you missed a blank.
- ✅ **Check** lines tell you whether your result matches the profiled data.
- Stuck for more than 10 minutes? Open `solutions/01_bronze_ingest_solution.ipynb`. Read it, close it, then type your own version. Don't paste.

Run cells top to bottom with **Shift + Enter**.

## 🧠 Why we're doing this: the medallion architecture

Databricks projects organise data in three layers:

| Layer | What it holds | Rule | Excel equivalent |
|-------|---------------|------|------------------|
| **Bronze** | The raw data, exactly as it arrived | Never edit, only add | The raw export tab you never touch |
| **Silver** | Cleaned, typed, standardised data | One clean row per real-world event | Your Power Query output table |
| **Gold** | Business-ready tables and aggregates | Built for a specific question or dashboard | The table that feeds your PivotTables |

**Why not clean the data while loading it?** If a cleaning rule turns out to be wrong (and some will), you fix the rule and re-run silver from bronze in minutes. You don't have to download 1.16 GB again and hope the source hasn't changed. Bronze is your single source of truth and your audit trail.

🎤 **Interview angle:** *"Why keep a raw layer?"* → For reproducibility and auditing: every number in a dashboard can be traced back to the untouched source, and cleaning logic can change without re-extracting.

## Setup
These two cells are written for you. Read them, then run them.

In [0]:
# Where everything lives (Unity Catalog: catalog.schema.object)
CATALOG = "workspace"
SCHEMA  = "dubai_re"
VOLUME  = "raw_files"

RAW_PATH     = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/"
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_transactions"

print("Raw files folder:", RAW_PATH)
print("Bronze table:    ", BRONZE_TABLE)

Raw files folder: /Volumes/workspace/dubai_re/raw_files/
Bronze table:     workspace.dubai_re.bronze_transactions


In [0]:
# A tiny helper that compares your result with the profiled data
def check(label, actual, expected):
    fmt = (lambda v: f"{v:,}") if isinstance(expected, int) else str
    if actual == expected:
        print(f"✅ {label}: {fmt(actual)}")
    else:
        print(f"❌ {label}: got {actual}, expected {fmt(expected)}. Re-read the hint above this cell.")

## Step 1: Look before you load

🧠 **Why we're doing this**
Always confirm what you're about to load: how many files, how big. If someone dropped a third file in the folder, or an upload stopped halfway, you want to catch it *now*, not after a dashboard shows the wrong totals.

`dbutils` is Databricks' built-in toolbox for files and the workspace. `dbutils.fs.ls(path)` is the equivalent of `ls` in a terminal, or of opening the folder in Finder.

✏️ **TODO 1:** list the files in `RAW_PATH`.
*Hint:* `dbutils.fs.ls(<path>)` returns a list of `FileInfo` objects, each with `.name` and `.size` (in bytes).

In [0]:
# TODO 1: list the files in RAW_PATH
files = dbutils.fs.ls(RAW_PATH)

for f in files:
    print(f"{f.name:<45} {f.size / 1_000_000:,.0f} MB")

csv_files = [f for f in files if f.name.endswith(".csv")]
check("CSV files found", len(csv_files), 2)

transactions_2026-11-06_21-44-39_0001.csv     582 MB
transactions_2026-11-06_21-44-39_0002.csv     582 MB
✅ CSV files found: 2


## Step 2: Read both CSVs into one DataFrame

🧠 **Why we're doing this**

**A Spark DataFrame is like a pandas DataFrame, but split across many machines.** The syntax looks similar. The big difference is *when* the work happens.

**Lazy evaluation.** `spark.read...load(...)` doesn't read 1.16 GB. It just writes down a plan, the way Power Query records your applied steps without running them until you click **Close & Load**. Spark has two kinds of commands:
- **Transformations** (read, filter, select, withColumn): add a step to the plan. Instant.
- **Actions** (count, display, write): actually run the plan. These take time.

**Why every column stays text (string) in bronze.** Spark *could* guess the types with `inferSchema`, but:
1. To guess, it reads the whole 1.16 GB once just to look, and then again to load.
2. It guesses wrong in ways that lose information. For example, `project_number` is stored as `"1960.00"`, and an ID column of digits could lose leading zeros.
3. Bronze should be the data *as it arrived*. Choosing types is a cleaning decision, so it belongs in silver.

**Wildcards.** `transactions_*.csv` matches both files, so you get one DataFrame without looping over files yourself.

✏️ **TODO 2:** read the CSVs.
- The first line of each file is a header, so set the `"header"` option to `"true"`.
- Load the path `RAW_PATH + "transactions_*.csv"`.

In [0]:
# TODO 2: read both CSV files into one DataFrame
df_raw = (
    spark.read
         .format("csv")
         .option("header", True)
         .load(RAW_PATH + "transactions_*.csv")
)

check("Columns", len(df_raw.columns), 47)

✅ Columns: 47


Notice that the cell above ran almost instantly. That's lazy evaluation: Spark only read the header line to learn the column names. Nothing else has been read yet.

## Step 3: Inspect the structure

🧠 **Why we're doing this**
`printSchema()` shows each column and its type, like `\d table_name` in psql or the column list in pgAdmin. `display()` is Databricks' interactive table viewer: you can sort it and turn it into a chart.

✏️ **TODO 3a:** print the schema of `df_raw`.
✏️ **TODO 3b:** show 5 rows of a few useful columns. Use `.select(...)` to choose columns and `.limit(5)` to keep it small.

In [0]:
# TODO 3a: print the schema
df_raw.printSchema()

root
 |-- actual_worth: string (nullable = true)
 |-- area_id: string (nullable = true)
 |-- area_name_ar: string (nullable = true)
 |-- area_name_en: string (nullable = true)
 |-- building_name_ar: string (nullable = true)
 |-- building_name_en: string (nullable = true)
 |-- has_parking: string (nullable = true)
 |-- instance_date: string (nullable = true)
 |-- master_project_ar: string (nullable = true)
 |-- master_project_en: string (nullable = true)
 |-- meter_rent_price: string (nullable = true)
 |-- meter_sale_price: string (nullable = true)
 |-- nearest_landmark_ar: string (nullable = true)
 |-- nearest_landmark_en: string (nullable = true)
 |-- nearest_mall_ar: string (nullable = true)
 |-- nearest_mall_en: string (nullable = true)
 |-- nearest_metro_ar: string (nullable = true)
 |-- nearest_metro_en: string (nullable = true)
 |-- no_of_parties_role_1: string (nullable = true)
 |-- no_of_parties_role_2: string (nullable = true)
 |-- no_of_parties_role_3: string (nullable = true

In [0]:
# TODO 3b: show 5 rows of these columns
cols_to_look_at = ["transaction_id", "instance_date", "trans_group_en", "procedure_name_en",
                   "reg_type_en", "area_name_en", "actual_worth", "procedure_area"]

display(df_raw.select(cols_to_look_at).limit(5))

transaction_id,instance_date,trans_group_en,procedure_name_en,reg_type_en,area_name_en,actual_worth,procedure_area
2-13-2011-1692,2011-03-14,Mortgages,Mortgage Registration,Existing Properties,Al Barshaa South Second,875000.00,138.27
1-11-2022-16850,2022-07-19,Sales,Sell,Existing Properties,Al Hebiah Fourth,980000.00,130.42
2-13-2003-404,2003-05-27,Mortgages,Mortgage Registration,Existing Properties,Um Al Sheif,532000.00,1393.55
1-102-2018-16683,2018-12-17,Sales,Sell - Pre registration,Off-Plan Properties,Business Bay,1350000.00,83.75
1-41-2025-31617,2025-12-17,Sales,Delayed Sell,Existing Properties,Al Yufrah 1,5481888.00,429.46


🤔 **Think about it** (answer in a markdown cell below or in your notes; answers are in the solution notebook):
1. Every column shows as `string`. Why is that exactly what we want in bronze?
2. The attribute sheet lists 46 columns, but we have 47. Which one is extra, and what might it be useful for?
3. Look at `instance_date`. Does it match the format the attribute sheet says (DD-MM-YYYY)?

## Step 4: Count the rows (your first action)

🧠 **Why we're doing this**
`.count()` is an **action**, so this is the moment Spark actually reads all 1.16 GB. On Free Edition's serverless compute, expect roughly 20–60 seconds.

Comparing the row count with a number you trust is the most basic data quality check there is. Here we compare with the DuckDB profile in `docs/data_profile.md`. In a job, you'd compare with the row count the source system reports.

✏️ **TODO 4:** count the rows in `df_raw` and store the result in `row_count`.

In [0]:
# TODO 4: count the rows
row_count = df_raw.count()

check("Rows in raw CSVs", row_count, 1_814_844)

✅ Rows in raw CSVs: 1,814,844


## Step 5: Add audit (lineage) columns

🧠 **Why we're doing this**
In six months someone will ask: *"This transaction looks wrong. Which file did it come from, and when did we load it?"* Two extra columns answer that:

| Column | Source | Meaning |
|--------|--------|---------|
| `_source_file` | `_metadata.file_name`, a hidden column Spark attaches to every row read from a file | Which CSV the row came from |
| `_ingested_at` | `F.current_timestamp()` | When this load ran |

The leading underscore is a common convention: *"this column was added by the pipeline, not the source."*

`withColumn(name, expression)` adds or replaces one column. It's the Spark version of adding a custom column in Power Query, or `SELECT *, expression AS name` in SQL.

`F` is the standard short name for `pyspark.sql.functions`, where all the built-in functions (`col`, `lit`, `current_timestamp`, `upper` and so on) live.

✏️ **TODO 5:** add the two columns.
*Hint:* `F.col("_metadata.file_name")` and `F.current_timestamp()`

In [0]:
from pyspark.sql import functions as F

# TODO 5: add the two audit columns
df_bronze = (
    df_raw
      .withColumn("_source_file", F.col("_metadata.file_name"))
      .withColumn("_ingested_at", F.current_timestamp())
)

check("Columns after audit", len(df_bronze.columns), 49)
check("Distinct source files", df_bronze.select("_source_file").distinct().count(), 2)

✅ Columns after audit: 49
✅ Distinct source files: 2


## Step 6: Save as a Delta table

🧠 **Why we're doing this**

**Delta Lake** is the table format Databricks uses. Underneath it's still files (Parquet), plus a transaction log that adds database-like features:

| Delta feature | What it means | Where you've seen it |
|---------------|---------------|----------------------|
| **ACID transactions** | A write fully succeeds or fully fails; nobody sees half a table | `BEGIN … COMMIT` in PostgreSQL |
| **Time travel** | Every version of the table is kept and can be queried | Version history in Excel / Google Sheets |
| **Schema enforcement** | Appending data with the wrong columns fails loudly | A typed column rejecting text |

**`saveAsTable`** registers the table in Unity Catalog, so you (and SQL, and Power BI later) can find it by name instead of by file path.

**`mode("overwrite")`** replaces the table each run. That makes the notebook **idempotent**: run it once or ten times and you get the same result, never duplicated rows. (`mode("append")` would add another 1.8M rows every run.)

✏️ **TODO 6:** write `df_bronze` as a Delta table.
- format: `"delta"`
- mode: `"overwrite"`
- save as: `BRONZE_TABLE`

This is the slow cell: Spark reads the CSVs again and writes compressed Delta files. Allow 1–3 minutes.

In [0]:
# TODO 6: save as a Delta table
(
    df_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")   # lets a re-run replace the table even if columns change
        .saveAsTable(BRONZE_TABLE)
)

check("Rows in bronze table", spark.table(BRONZE_TABLE).count(), 1_814_844)

✅ Rows in bronze table: 1,814,844


💡 Go to **Catalog → workspace → dubai_re**. Your `bronze_transactions` table is now there, with columns, sample data and history, without writing any more code. Compare the size of the table with the 1.16 GB of CSV. Delta/Parquet compresses column by column, so the table should be several times smaller.

## Step 7: Query it with SQL

🧠 **Why we're doing this**
In a Databricks notebook, Python and SQL work on the same tables. Start a cell with `%sql` and the whole cell runs as SQL, with the results shown as a table you can chart. Most analysts on Databricks switch back and forth: Python for pipelines, SQL for questions.

✏️ **TODO 7:** in the `%sql` cell below, write a query that counts transactions per `trans_group_en`, sorted from most to least. You've written this query on Chinook many times; only the table name is new.

In [0]:
%sql
-- TODO 7: count transactions per trans_group_en, most first
SELECT trans_group_en, 
        COUNT(*) AS transactions
FROM workspace.dubai_re.bronze_transactions
GROUP BY trans_group_en
ORDER BY transactions DESC

trans_group_en,transactions
Sales,1387509
Mortgages,359126
Gifts,68209


This check cell runs the same query from Python with `spark.sql(...)`, which returns a DataFrame. That's how you'd use SQL inside a Python pipeline.

In [0]:
rows = spark.sql(f"""
    SELECT trans_group_en, COUNT(*) AS transactions
    FROM {BRONZE_TABLE}
    GROUP BY trans_group_en
""").collect()

by_group = {r["trans_group_en"]: r["transactions"] for r in rows}
check("Sales",     by_group.get("Sales"),     1_387_509)
check("Mortgages", by_group.get("Mortgages"), 359_126)
check("Gifts",     by_group.get("Gifts"),     68_209)

✅ Sales: 1,387,509
✅ Mortgages: 359,126
✅ Gifts: 68,209


⭐ **Stretch TODO 7b:** which year has the most transactions?
In bronze, `instance_date` is still text (`"2025-03-14"`), so take the first 4 characters: `LEFT(instance_date, 4)`.
Expected answer: **2025, with 266,991 transactions**. Then look at the earliest "years" in the result. Something odd is going on there (see DQ2 in `docs/data_profile.md`).

In [0]:
%sql
-- Stretch TODO 7b: transactions per year, most first
SELECT LEFT(instance_date, 4) AS year, 
        COUNT(*) AS transactions
FROM workspace.dubai_re.bronze_transactions
GROUP BY year
ORDER BY transactions DESC

year,transactions
2025,266991
2024,224395
2026,180708
2023,165260
2022,120014
2021,82822
2009,79189
2013,77972
2014,70373
2017,64548


## Step 8: Look at the table's history (time travel)

🧠 **Why we're doing this**
Every write to a Delta table creates a new version. `DESCRIBE HISTORY` lists them: who wrote, when, and how many rows. Re-run Step 6 and you'll see version 1 appear. You can query an old version with `SELECT * FROM table VERSION AS OF 0`, which is useful when a bad load needs undoing.

Nothing to write here. Just run it.

In [0]:
%sql
DESCRIBE HISTORY workspace.dubai_re.bronze_transactions

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-11-12T03:59:56.000Z,72957123150992,marzrodie@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""84a5cf56-1d44-4f16-90a7-3e1d1bac9b7a"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-a1e4305a-a519-4bb9-8fa2-1a61c2caf98e"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-f423eea0-d669-41d7-b562-dd234ca23e8a""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(2738419370374728),a2543f00-7ce8-4fd2-a3c4-1a22fd673a5e,1112-033154-4czf9wps-v2n,null,WriteSerializable,false,"Map(numFiles -> 10, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 1814844, numOutputBytes -> 66558751)",null,Databricks-Runtime/19.14.x-aarch64-photon-scala2.13


## ✅ Module 1 complete

**What you built:** a bronze Delta table of 1,814,844 raw transactions with lineage columns, loaded by a notebook you can safely re-run.

**🎤 Practise saying these out loud** (model answers in `LEARNING.md`):
1. What is the medallion architecture, and why keep a bronze layer?
2. What's the difference between a transformation and an action in Spark?
3. Why didn't you use `inferSchema`?
4. What does Delta Lake add on top of Parquet files?
5. What does "idempotent" mean, and how did you make this notebook idempotent?

**Commit your work**
1. In Databricks: **File → Export → IPython Notebook**
2. Save it over `databricks/01_bronze_ingest.ipynb` in your repo folder (keep the outputs; recruiters like seeing results).
3. In the VS Code terminal:
```bash
git add databricks/01_bronze_ingest.ipynb
git commit -m "Add bronze ingestion: 1.81M raw transactions into Delta with lineage columns"
git push
```
4. Tick Module 1 in the README roadmap.

**Next:** Module 2, Silver. Typing, the date fix, standardising rooms and communities, and outlier flags.